In [0]:
dbutils.widgets.text("path","")

In [0]:
import json
from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import read_s3_parquet,write_raw

#0.logger
logger = get_logger("s3-ingestion")


# 1  config path
dbutils.widgets.text("path","")
config_path= dbutils.widgets.get("path")

# 2 reading config 

with open(config_path,"r") as f:
    config = json.load(f)

# 3 Configuration

source_config= config["source"]
target_config=config["target"]
write_options=config["write_options"]

# 4 run date

run_date = date.today().isoformat()




aws_secret_access_key=dbutils.secrets.get(scope="retail-platform-dev",key=source_config["aws_secret_access_key"])
aws_access_key_id=dbutils.secrets.get(scope="retail-platform-dev",key=source_config["aws_access_key_id"])

#5 read  data from sql server
df=read_s3_parquet(spark,aws_access_key_id ,aws_secret_access_key,source_config["bucket"],source_config["folder"],source_config["file"])
logger.info("read  %s rows",df.count())
logger.info("sample data...........")
df.show()


#6 write from sql server

target_path=f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"

write_raw(df,target_path,target_config["file_format"],target_config["mode"],write_options)

logger.info("data landed at %s",target_path)